# uav-traffic-vision — YOLO26s-obb on DOTA-v1.0 (Colab training, Phase 3)

One-click training notebook: **Runtime → Run all**.

Phase 3 supplement to the main VisDrone track: same aerial-imagery domain, a different
annotation format — **oriented bounding boxes (OBB)** instead of axis-aligned boxes.

What it does:
1. Downloads DOTA-v1.0 (~2 GB, 1,411 train / 458 val images, 15 classes) and tiles it
   into overlapping 1024x1024 crops with `split_dota` — DOTA images run up to ~13,000 px
   on a side and can't be fed to YOLO directly (see `reports/dota_stats.md` for the
   measured size distribution; the local repo's tiling run produced 15,749 train /
   5,297 val tiles from the 1,869 raw images).
2. **Probes training cost first**: this recipe (yolo26s-obb, 15,749 tiles, imgsz=1024)
   has never been timed end-to-end. A 2-epoch throwaway run measures real epoch time
   and prints a projected total-hours / compute-unit cost for the full 100-epoch run —
   same safety net as the VisDrone imgsz=1024 experiment.
3. **Full training only starts after you review the estimate** and set `RUN_FULL = True`:
   `yolo26s-obb` at **imgsz=1024** (the tile size — no separate resolution comparison
   here, unlike the VisDrone track).
4. Checkpoints land directly on **Google Drive** (`MyDrive/uav-traffic-vision/runs/...`),
   so a disconnect never loses progress — see the resume note below.
5. Validates `best.pt` (built-in rotated-IoU OBB mAP) and copies it to a stable Drive path.

Requirements: Colab Pro, GPU runtime (**L4** or **A100** — Runtime → Change runtime type).

In [ ]:
# GPU / CPU check
import os
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU — Runtime -> Change runtime type -> GPU"
GPU_NAME = torch.cuda.get_device_name(0)
print(f"CPU cores: {os.cpu_count()}")

In [ ]:
# Mount Google Drive (checkpoints and final weights live here)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Configuration
from pathlib import Path

MODEL    = "yolo26s-obb.pt"
IMGSZ    = 1024  # = split_dota's tile size; DOTA's native training resolution
EPOCHS   = 100
PATIENCE = 20

# batch size on A100 was swept empirically via the probe cell below (2-epoch
# throwaway runs on the real, full split dataset), not guessed:
#   batch=24: 4.8 min/epoch, ~120 CU projected -- clean
#   batch=40: 4.5 min/epoch, ~114 CU projected -- clean
#   batch=64: 4.3 min/epoch, ~108 CU projected -- clean, best result
#   batch=96: repeated "CUDA OutOfMemoryError in TaskAlignedAssigner, using CPU"
#             warnings, GPU_mem oscillating 75.5-78.4G/80G -- rejected: too close
#             to the ceiling to be safe over a full 100-epoch run (batches vary a
#             lot in objects/image, e.g. 181 vs 598 instances back at batch=24),
#             and the CPU-fallback path was likely eating the speed gain anyway
# Gains were already flattening (24->40 saved 6.3%, 40->64 only 4.4% more), so 64
# was taken as the sweet spot rather than chasing the exact ceiling between 64-96.
# L4 was NOT re-tested this round; its value is still the original conservative
# guess, unverified -- rerun the same probe sweep if training on L4.
if "A100" in GPU_NAME:
    BATCH = 64
elif "L4" in GPU_NAME:
    BATCH = 8
else:
    BATCH = 8  # conservative default for an unrecognized GPU

CACHE    = "disk"  # cache decoded tiles on local disk (same win as the VisDrone notebook)
SEED     = 42
RESUME   = False   # after a disconnect: set True, Runtime -> Run all again

DRIVE_ROOT  = Path("/content/drive/MyDrive/uav-traffic-vision")
RUNS_DIR    = DRIVE_ROOT / "runs"     # ultralytics project dir -> checkpoints on Drive
WEIGHTS_DIR = DRIVE_ROOT / "weights"  # stable copies of best.pt / results.csv
RUN_NAME    = f"dota_obb_yolo26s_{IMGSZ}"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"GPU={GPU_NAME}  BATCH={BATCH}  cache={CACHE}")

In [ ]:
# Install ultralytics + shapely (split_dota's bbox_iof needs it), pinned to the
# versions locked in the repo's uv.lock
%pip install -q ultralytics==8.4.90 shapely==2.1.2

from ultralytics import settings
settings.update({"datasets_dir": "/content/datasets"})  # Colab local disk, NOT Drive (fast I/O)

In [ ]:
# Download DOTA-v1.0 raw imagery (~2 GB on first run; images up to ~13,000 px per side)
from ultralytics.data.utils import check_det_dataset

data = check_det_dataset("DOTAv1.yaml", autodownload=True)
print("train:", data["train"])
print("val:  ", data["val"])

In [ ]:
# Tile into overlapping 1024x1024 crops (single-scale: split_dota's own defaults,
# crop=1024 gap=200 rate=1.0 -- multiscale rates=[0.5,1,1.5] would ~triple both
# preprocessing time and training-set size for a further mAP gain; left as a
# documented option, not the default, to control compute cost)
import yaml
from ultralytics.data.split_dota import split_trainval

DOTA_ROOT  = Path("/content/datasets/DOTAv1")
SPLIT_ROOT = Path("/content/datasets/DOTAv1-split")

if not (SPLIT_ROOT / "images" / "train").exists():
    split_trainval(data_root=str(DOTA_ROOT), save_dir=str(SPLIT_ROOT))

n_train = len(list((SPLIT_ROOT / "images" / "train").glob("*")))
n_val   = len(list((SPLIT_ROOT / "images" / "val").glob("*")))
print(f"tiles: train={n_train}  val={n_val}")

CLASS_NAMES = [
    "plane", "ship", "storage tank", "baseball diamond", "tennis court",
    "basketball court", "ground track field", "harbor", "bridge",
    "large vehicle", "small vehicle", "helicopter", "roundabout",
    "soccer ball field", "swimming pool",
]
data_yaml = {
    "path": str(SPLIT_ROOT), "train": "images/train", "val": "images/val",
    "names": dict(enumerate(CLASS_NAMES)),
}
DOTA_YAML = Path("/content/dota_obb.yaml")
DOTA_YAML.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
print(f"wrote {DOTA_YAML}")

## Estimate training cost (2-epoch probe)

Trains 2 throwaway epochs on the real, full split dataset (not a subset) so the
measured per-epoch time is representative, then projects the total time and
compute-unit cost of the full 100-epoch run. Review the printout before deciding
whether to flip `RUN_FULL = True` in the next section.

In [ ]:
# Estimate full-training cost first (2-epoch probe, throwaway run on local disk).
# This recipe has never been timed end-to-end: 15,749 train tiles is more images/
# epoch than VisDrone's 6,471, and the OBB head adds an angle-regression branch on
# top of plain detection -- so don't commit to 100 epochs blind. Same safety net as
# the VisDrone imgsz=1024 experiment.
ESTIMATE_FIRST = True   # set False to skip the ~5-10 min probe

import time

from ultralytics import YOLO

if "A100" in GPU_NAME:
    CU_PER_HOUR = 15.0  # verify actual rate: View resources in Colab
elif "L4" in GPU_NAME:
    CU_PER_HOUR = 2.4
else:
    CU_PER_HOUR = 2.4

if ESTIMATE_FIRST:
    import pandas as pd

    probe_dir = Path("/content/probe_runs")
    t0 = time.time()
    YOLO(MODEL).train(
        data=str(DOTA_YAML), imgsz=IMGSZ, epochs=2, batch=BATCH, seed=SEED,
        project=str(probe_dir), name="probe", exist_ok=True,
        val=False, plots=False, save=False,
    )
    wall = time.time() - t0
    try:  # per-epoch time from results.csv (2nd epoch, excludes warmup)
        t = pd.read_csv(probe_dir / "probe" / "results.csv")["time"]
        epoch_s = float(t.iloc[1] - t.iloc[0])
    except Exception:
        epoch_s = wall / 2
    total_h = epoch_s * EPOCHS / 3600
    print(f"\nmeasured epoch time (batch={BATCH}) on {GPU_NAME}: {epoch_s / 60:.1f} min")
    print(f"projected full run, {EPOCHS} epochs (no early stop): ~{total_h:.1f} h "
          f"~= {total_h * CU_PER_HOUR:.0f} compute units @ {CU_PER_HOUR}/h")
    print(f"patience={PATIENCE} usually stops earlier, so treat this as an upper bound.")
    print("If acceptable: set RUN_FULL = True in the next cell and run it.")
else:
    print("ESTIMATE_FIRST = False -- probe skipped")

## Full training — only runs after you review the estimate above

**If Colab disconnects mid-run**: reconnect, set `RESUME = True` in the configuration
cell, then Runtime → Run all — training continues from `last.pt`. `patience=20` stops
early if the model converges before 100 epochs, same as both VisDrone runs did.

In [ ]:
RUN_FULL = False  # review the probe estimate above, then set True and run this cell

if RUN_FULL:
    last_pt = RUNS_DIR / RUN_NAME / "weights" / "last.pt"
    if RESUME and last_pt.exists():
        print(f"resuming from {last_pt}")
        model = YOLO(str(last_pt))
        model.train(resume=True)
    else:
        model = YOLO(MODEL)
        t0 = time.time()
        model.train(
            data=str(DOTA_YAML), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE,
            batch=BATCH, cache=CACHE, seed=SEED, project=str(RUNS_DIR), name=RUN_NAME,
            exist_ok=True,
        )
        print(f"training wall time: {(time.time() - t0) / 3600:.2f} h")
else:
    print("RUN_FULL = False -- skipped (review the probe estimate above first)")

In [ ]:
# Validate the best checkpoint (built-in rotated-IoU OBB mAP -- no separate
# pycocotools pipeline needed here, unlike the VisDrone axis-aligned-box track)
best_pt = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
if best_pt.exists():
    metrics = YOLO(str(best_pt)).val(data=str(DOTA_YAML), imgsz=IMGSZ, split="val")
    print(f"val mAP50={metrics.box.map50:.4f}  mAP50-95={metrics.box.map:.4f}")
else:
    print(f"{best_pt} not found -- set RUN_FULL = True in the previous cell and rerun.")

In [ ]:
# Copy artifacts to a stable Drive path
import shutil

if best_pt.exists():
    shutil.copy2(best_pt, WEIGHTS_DIR / "yolo26s_dota_1024.pt")
    results_csv = RUNS_DIR / RUN_NAME / "results.csv"
    if results_csv.exists():
        shutil.copy2(results_csv, WEIGHTS_DIR / f"results_{RUN_NAME}.csv")
    print("artifacts on Drive:")
    for p in sorted(WEIGHTS_DIR.iterdir()):
        print(" ", p)
    print("\nNext: download yolo26s_dota_1024.pt into the repo's weights/ folder.")
else:
    print("nothing to copy yet -- run full training first.")